# Tutorial 5: Select telluric fitting regions interactively

This tutorial creates a reusable fit-region file with PyMolFit's interactive selector. Candidate AER transitions are marked by molecule, strong-line windows can be proposed automatically, and manual fit or exclusion rectangles can be added before saving.

## Install interactive plotting support

Use the same environment for PyMolFit and the notebook kernel:

```bash
python -m pip install "pymolfit[interactive]"
```

In [ ]:
%matplotlib widget

from pathlib import Path

from pymolfit import (
    RegionSelection,
    correct,
    load_region_file,
    load_spectrum,
    plot_fit,
    select_telluric_regions,
)

## Load the example spectrum

The selected regions are stored in a separate ECSV file. Its metadata records the wavelength unit and air/vacuum medium, allowing `correct()` to convert the regions safely when needed.

In [ ]:
candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the PyMolFit repository or tutorials directory"
    )

INPUT = TUTORIAL_ROOT / "data" / "ADP.2017-04-07T01_04_41.632.fits"
REGION_FILE = Path.cwd() / "tutorial_telluric_regions.ecsv"

spectrum = load_spectrum(
    INPUT,
    wavelength_medium="air",
).to_air().to_unit("angstrom")

print(f"Pixels: {spectrum.wavelength.size:,}")
print(f"Region file: {REGION_FILE}")

## Open or reuse the selector

On the first run, the interactive window opens. Enter a number in **Lines** and press **Automatic** to propose windows around that many strongest covered AER transitions. Proposals inside detector or order gaps are skipped, and overlapping windows are merged.

To edit manually, choose **Fit**, **Exclude**, or **Delete**, enable **Draw regions**, and drag wavelength rectangles. **Undo** reverses the latest edit, **Clear** removes all regions, and **Save All** writes the complete ECSV file.

If the ECSV file already exists, PyMolFit loads it and skips the selector window.

In [ ]:
selection_or_selector = select_telluric_regions(
    spectrum,
    output_path=REGION_FILE,
    automatic_region_count=100,
)

if isinstance(selection_or_selector, RegionSelection):
    print(f"Reused {len(selection_or_selector.fit_ranges)} fit regions")
else:
    print("Select regions and press Save All before continuing.")

## Inspect the saved selection

Run this cell after pressing **Save All**. The file can contain multiple fit and exclusion regions.

In [ ]:
if not REGION_FILE.is_file():
    raise FileNotFoundError(
        "No region file exists yet. Press Save All in the selector first."
    )

selection = load_region_file(REGION_FILE)
print("Fit regions:", selection.fit_ranges)
print("Excluded regions:", selection.exclude_ranges)
print("Coordinates:", selection.wavelength_medium, selection.wavelength_unit)

## Correct the spectrum with the saved regions

Fit regions determine which observed pixels estimate atmospheric and instrumental parameters. Exclusion regions override fit regions during parameter estimation. The fitted transmission is still evaluated and applied over the complete spectrum.

In [ ]:
result = correct(
    input_path=INPUT,
    wavelength_medium="air",
    region_file=REGION_FILE,
)

if not result.success:
    raise RuntimeError(result.message)

In [ ]:
plot_fit(result)

## Reopen an existing selection

To change an existing file, reopen it with `reuse_existing=False`. Its saved regions are loaded into the interface instead of starting from an empty selection.

```python
edit_selector = select_telluric_regions(
    spectrum,
    output_path=REGION_FILE,
    reuse_existing=False,
)
```